In [16]:
!pip -q install timm gdown scikit-learn captum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 15.8 MB/s eta 0:00:00


In [11]:
from google.colab import drive
import os
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive"
print("Google Drive mounted at:")
print(DRIVE_ROOT)
ROOT = None

print("\nSearching for dataset folders...")

for current_path, directories, files in os.walk(DRIVE_ROOT):

    folder_names = set(directories)

    if {"train", "val", "test"}.issubset(folder_names):
        ROOT = current_path
        break
if ROOT is None:

    print("\nCould not automatically find train/val/test folders.")

    print("\nFolders available directly inside MyDrive:")

    for item in os.listdir(DRIVE_ROOT):
        print("  ", item)

    raise FileNotFoundError(
        "\nCould not find a folder containing train, val, and test."
    )

print("\nDataset found successfully!")
print("Dataset root:")
print(ROOT)
TRAIN_DIR = os.path.join(ROOT, "train")
VAL_DIR   = os.path.join(ROOT, "val")
TEST_DIR  = os.path.join(ROOT, "test")

print("\nTrain:", TRAIN_DIR)
print("Val:  ", VAL_DIR)
print("Test: ", TEST_DIR)
for split_name, split_path in [
    ("TRAIN", TRAIN_DIR),
    ("VAL", VAL_DIR),
    ("TEST", TEST_DIR)
]:

    print(f"\n{split_name}")

    classes = sorted([
        d for d in os.listdir(split_path)
        if os.path.isdir(os.path.join(split_path, d))
    ])

    print("Classes:", classes)

    for cls in classes:

        class_path = os.path.join(split_path, cls)

        image_count = len([
            f for f in os.listdir(class_path)
            if os.path.isfile(os.path.join(class_path, f))
        ])

        print(f"  {cls}: {image_count} images")
MODEL_NAME = "swin_base_patch4_window7_224"
PRETRAINED = True

IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 100

LR = 1e-4
WEIGHT_DECAY = 0.05

PATIENCE = 10
SEED = 42
NUM_WORKERS = 2

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted at:
/content/drive/MyDrive

Searching for dataset folders...

Dataset found successfully!
Dataset root:
/content/drive/MyDrive/skin dataset/MILK10k_Split

Train: /content/drive/MyDrive/skin dataset/MILK10k_Split/train
Val:   /content/drive/MyDrive/skin dataset/MILK10k_Split/val
Test:  /content/drive/MyDrive/skin dataset/MILK10k_Split/test

TRAIN
Classes: ['images']
  images: 8384 images

VAL
Classes: ['images']
  images: 1048 images

TEST
Classes: ['images']
  images: 1048 images


## Cell 3 – Build the 5-class dataframe

In [12]:
import pandas as pd, numpy as np, torch, random, glob, os

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(!! turn on GPU runtime)")

CLASSES = ["melanoma", "nevus", "bcc", "scc", "keratoses"]
CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}

def map_label(name):
    s = str(name).lower()
    if "collision" in s:            return None
    if "melanoma" in s:             return "melanoma"
    if "squamous cell carcinoma" in s: return "scc"
    if "basal cell carcinoma" in s: return "bcc"
    if "keratosis" in s:            return "keratoses"
    if "nevus" in s:                return "nevus"
    return None

def load_split(split):
    split_dir = os.path.join(ROOT, split)
    csv_candidates = glob.glob(os.path.join(split_dir, "*Supplement_modified*"))
    if not csv_candidates:
        csv_candidates = glob.glob(os.path.join(split_dir, "*.csv"))

    csv = csv_candidates[0]
    print(f"Loading {split} from: {os.path.basename(csv)}")
    df = pd.read_csv(csv)
    df["cls"] = df["diagnosis_full"].map(map_label)
    n_all = len(df)
    df = df.dropna(subset=["cls"]).copy()
    df["path"] = df["isic_id"].apply(lambda i: os.path.join(split_dir, "images", f"{i}.jpg"))
    df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)
    df["label"] = df["cls"].map(CLASS2IDX)
    print(f"{split:5s}: kept {len(df)} / {n_all} images | {df["cls"].value_counts().to_dict()}")
    return df

train_df, val_df, test_df = load_split("train"), load_split("val"), load_split("test")

Device: cuda Tesla T4
Loading train from: Supplement_modified train.csv
train: kept 7686 / 8384 images | {'bcc': 4052, 'nevus': 1154, 'keratoses': 948, 'scc': 802, 'melanoma': 730}
Loading val from: Supplement_modified val.csv
val  : kept 970 / 1048 images | {'bcc': 510, 'nevus': 162, 'keratoses': 134, 'melanoma': 86, 'scc': 78}
Loading test from: Supplement_modified.csv
test : kept 964 / 1048 images | {'bcc': 482, 'nevus': 176, 'keratoses': 120, 'scc': 102, 'melanoma': 84}


## Cell 4 – Dataset, transforms, loaders

In [13]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
    T.ColorJitter(0.2, 0.2, 0.2, 0.02),
    T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

class LesionDS(Dataset):
    def __init__(self, df, tf): self.df, self.tf = df.reset_index(drop=True), tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.tf(Image.open(r["path"]).convert("RGB")), int(r["label"])

train_loader = DataLoader(LesionDS(train_df, train_tf), BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(LesionDS(val_df,   eval_tf),  BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(LesionDS(test_df,  eval_tf),  BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

## Cell 5 – Swin Transformer model

In [7]:
import timm

class SwinClassifier(torch.nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.backbone = timm.create_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=num_classes, drop_path_rate=0.1)
    def forward(self, x): return self.backbone(x)

model = SwinClassifier(len(CLASSES)).to(device)
print(f"{MODEL_NAME} | pretrained={PRETRAINED} | params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
if not PRETRAINED: print("!! Training from random weights will give poor accuracy.")

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

swin_base_patch4_window7_224 | pretrained=True | params: 86.7M


## Cell 6 – Metrics + Grad-CAM

In [14]:
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix)
import torch.nn.functional as F

def compute_metrics(y_true, y_pred, loss):
    mp, mr, mf, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=range(len(CLASSES)), zero_division=0)
    return dict(loss=loss, acc=accuracy_score(y_true, y_pred), bal_acc=balanced_accuracy_score(y_true, y_pred),
                macro_p=mp, macro_r=mr, macro_f1=mf,
                per_class=pd.DataFrame({"precision": p, "recall": r, "f1": f, "support": s}, index=CLASSES),
                cm=confusion_matrix(y_true, y_pred, labels=range(len(CLASSES))))

def print_metrics(name, m, plot_cm=False):
    print(f"  [{name}] loss={m['loss']:.4f} | acc={m['acc']:.4f} | balanced_acc={m['bal_acc']:.4f} | "
          f"macro_P={m['macro_p']:.4f} | macro_R={m['macro_r']:.4f} | macro_F1={m['macro_f1']:.4f}")
    print(m["per_class"].round(4).to_string())
    print("  Confusion matrix (rows=true, cols=pred):")
    print(pd.DataFrame(m["cm"], index=[f"T_{c}" for c in CLASSES], columns=[f"P_{c}" for c in CLASSES]).to_string())
    if plot_cm:
        fig, ax = plt.subplots(figsize=(5, 4.2)); ax.imshow(m["cm"], cmap="Blues")
        ax.set_xticks(range(5)); ax.set_xticklabels(CLASSES, rotation=45); ax.set_yticks(range(5)); ax.set_yticklabels(CLASSES)
        for i in range(5):
            for j in range(5): ax.text(j, i, m["cm"][i, j], ha="center", va="center", color="red")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{name} confusion matrix"); plt.tight_layout(); plt.show()

class SwinGradCAM:


    def __init__(self, model):
        self.model = model
        self.layer = model.backbone.layers[-1].blocks[-1].norm1
    def __call__(self, x, class_idx=None):
        store = {}
        def fwd(m, i, o):
            store["a"] = o
            o.register_hook(lambda g: store.__setitem__("g", g))
        h = self.layer.register_forward_hook(fwd)
        self.model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            x = x.clone().requires_grad_(True)
            out = self.model(x)
            if class_idx is None: class_idx = out.argmax(1)
            out.gather(1, class_idx[:, None]).sum().backward()
        h.remove()
        a, g = store["a"], store["g"]
        if a.dim() == 3:
            n = int(a.shape[1] ** 0.5); a, g = a.reshape(-1, n, n, a.shape[-1]), g.reshape(-1, n, n, g.shape[-1])
        w = g.mean((1, 2), keepdim=True)       # channel weights
        cam = (w * a).sum(-1, keepdim=True).relu().permute(0, 3, 1, 2)   # (B,1,H,W)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        mn, mx = cam.amin((1, 2), keepdim=True), cam.amax((1, 2), keepdim=True)
        return ((cam - mn) / (mx - mn + 1e-8)).detach().cpu().numpy(), out.argmax(1).detach().cpu().numpy()

gradcam = SwinGradCAM(model)
_mean, _std = torch.tensor(MEAN).view(3, 1, 1), torch.tensor(STD).view(3, 1, 1)

def show_gradcam(df, n_per_class=2, title="Grad-CAM", save=None):
    model.eval()
    ds = LesionDS(df, eval_tf); rows = []
    for c in range(len(CLASSES)):
        idx = df.index[df["label"] == c].tolist()[:n_per_class]
        rows.append(idx)
    fig, axes = plt.subplots(2 * n_per_class, len(CLASSES), figsize=(3 * len(CLASSES), 6 * n_per_class / 1.0 * 0.5 * 2 / 2 + 1))
    axes = np.atleast_2d(axes)
    for c, idxs in enumerate(rows):
        for k, i in enumerate(idxs):
            img, y = ds[i]
            cam, pred = gradcam(img[None].to(device))
            rgb = (img * _std + _mean).clamp(0, 1).permute(1, 2, 0).numpy()
            heat = plt.cm.jet(cam[0])[..., :3]
            axes[2 * k, c].imshow(rgb); axes[2 * k + 1, c].imshow(0.55 * rgb + 0.45 * heat)
            axes[2 * k, c].set_title(f"true: {CLASSES[y]}", fontsize=9)
            axes[2 * k + 1, c].set_title(f"pred: {CLASSES[pred[0]]}", fontsize=9,
                                         color="green" if pred[0] == y else "red")
        for r in range(2 * n_per_class): axes[r, c].axis("off")
    fig.suptitle(title); plt.tight_layout()
    if save: plt.savefig(save, dpi=110, bbox_inches="tight")
    plt.show()

## Cell 7 – Train (all metrics + Grad-CAM printed every epoch)

In [1]:
import time, math, os
from torch.amp import autocast, GradScaler

# class-weighted loss (BCC dominates the data)
counts = train_df["label"].value_counts().reindex(range(len(CLASSES))).values.astype(float)
cls_w = torch.tensor((counts.sum() / (len(CLASSES) * counts)) ** 0.5, dtype=torch.float32).to(device)  # sqrt-inverse freq (gentler)
cls_w = cls_w / cls_w.mean()
print("class weights:", dict(zip(CLASSES, cls_w.cpu().numpy().round(3))))
criterion = torch.nn.CrossEntropyLoss(weight=cls_w, label_smoothing=0.05)
eval_criterion = torch.nn.CrossEntropyLoss(weight=cls_w)

head_p = list(model.backbone.get_classifier().parameters()); head_ids = {id(p) for p in head_p}
body_p = [p for p in model.parameters() if id(p) not in head_ids]
opt = torch.optim.AdamW([{"params": body_p, "lr": LR}, {"params": head_p, "lr": LR * 10}], weight_decay=WEIGHT_DECAY)
total_steps, warm = EPOCHS * len(train_loader), len(train_loader)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                          else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm))))
scaler = GradScaler(enabled=device.type == "cuda")

# Define checkpoint paths
CHECKPOINT_PATH = "/content/checkpoint_swin.pth"
BEST_MODEL_PATH = "/content/best_swin.pth"

start_epoch = 1
best_f1, bad, history = -1, 0, []

# Load resume checkpoint if it exists
if os.path.exists(CHECKPOINT_PATH):
    print(f"Found checkpoint at {CHECKPOINT_PATH}. Resuming training...")
    # Explicitly set weights_only=False to allow loading complex Python states like optimizer/scheduler configuration and training history
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    opt.load_state_dict(checkpoint['optimizer_state_dict'])
    sched.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_f1 = checkpoint['best_f1']
    bad = checkpoint['bad']
    history = checkpoint['history']
    print(f"Resuming from Epoch {start_epoch} (Best val macro-F1 so far: {best_f1:.4f})")

@torch.no_grad()
def evaluate(loader):
    model.eval(); ys, ps, tot = [], [], 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with autocast(device.type, enabled=device.type == "cuda"):
            out = model(x)
        tot += eval_criterion(out.float(), y).item() * len(y)
        ys += y.cpu().tolist(); ps += out.argmax(1).cpu().tolist()
    return compute_metrics(ys, ps, tot / len(loader.dataset))

# fixed validation images for per-epoch Grad-CAM
val_cam_df = val_df.groupby("label", group_keys=False).head(2).reset_index(drop=True)

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time(); model.train(); ys, ps, tot = [], [], 0.0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast(device.type, enabled=device.type == "cuda"):
            out = model(x); loss = criterion(out.float(), y)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * len(y); ys += y.cpu().tolist(); ps += out.argmax(1).cpu().tolist()
    tr = compute_metrics(ys, ps, tot / len(train_loader.dataset))
    va = evaluate(val_loader)
    print("=" * 100)
    print(f"EPOCH {epoch}/{EPOCHS}  ({time.time()-t0:.0f}s)  lr={opt.param_groups[0]['lr']:.2e}")
    print_metrics("TRAIN", tr); print_metrics("VAL", va, plot_cm=True)
    show_gradcam(val_cam_df, 2, title=f"Grad-CAM (val) - epoch {epoch}")

    history.append({"epoch": epoch, **{f"train_{k}": tr[k] for k in ["loss","acc","bal_acc","macro_f1"]},
                    **{f"val_{k}": va[k] for k in ["loss","acc","bal_acc","macro_f1"]}})

    if va["macro_f1"] > best_f1:
        best_f1, bad = va["macro_f1"], 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  >> saved best (val macro-F1={best_f1:.4f})")
    else:
        bad += 1

    # Save training state checkpoint at the end of each epoch for resuming
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
        'scheduler_state_dict': sched.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_f1': best_f1,
        'bad': bad,
        'history': history
    }, CHECKPOINT_PATH)
    print(f"  >> saved checkpoint for resume at {CHECKPOINT_PATH}")


NameError: name 'train_df' is not defined

## Cell 8 – Final test evaluation + Grad-CAM

In [20]:
# ============================================================
# FINAL TEST EVALUATION
# BEST SWIN CHECKPOINT + INTEGRATED GRADIENTS
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt
import torch


# ============================================================
# 1. CHECK BEST MODEL
# ============================================================

BEST_MODEL_PATH = "/content/best_swin.pth"

if not os.path.exists(BEST_MODEL_PATH):
    raise FileNotFoundError(
        f"Best model not found at: {BEST_MODEL_PATH}\n"
        "Make sure training has completed at least one epoch "
        "and a best checkpoint was saved."
    )


# ============================================================
# 2. LOAD BEST MODEL
# ============================================================

print("=" * 80)
print("Loading best model...")
print("=" * 80)

model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=device,
        weights_only=True
    )
)

model.to(device)


# ============================================================
# 3. FIND BEST EPOCH FROM HISTORY
# ============================================================

if len(history) == 0:
    raise ValueError(
        "Training history is empty. "
        "Cannot determine best epoch."
    )

h = pd.DataFrame(history)

best_idx = h["val_macro_f1"].idxmax()

best_epoch = int(
    h.loc[best_idx, "epoch"]
)

best_macro_f1 = float(
    h.loc[best_idx, "val_macro_f1"]
)

print("\n" + "=" * 80)
print("BEST VALIDATION RESULT")
print("=" * 80)

print(
    f"Best Epoch       : {best_epoch}"
)

print(
    f"Best Val Macro-F1: {best_macro_f1:.4f}"
)

print("=" * 80)


# ============================================================
# 4. TEST EVALUATION
# ============================================================

te = evaluate(
    test_loader
)

print(
    "\n"
    + "#" * 40
    + " TEST RESULTS (BEST CHECKPOINT) "
    + "#" * 40
)

print_metrics(
    "TEST",
    te,
    plot_cm=True
)


# ============================================================
# 5. TEST METRIC SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FINAL TEST METRICS")
print("=" * 80)

print(
    f"Test Loss       : {te['loss']:.4f}"
)

print(
    f"Test Accuracy   : {te['acc']:.4f}"
)

print(
    f"Test Balanced Acc: {te['bal_acc']:.4f}"
)

print(
    f"Test Macro-F1   : {te['macro_f1']:.4f}"
)

print("=" * 80)


# ============================================================
# 6. INTEGRATED GRADIENTS ON TEST IMAGES
# ============================================================

# Select fixed number of test images per class
test_ig_df = (
    test_df
    .groupby(
        "label",
        group_keys=False
    )
    .head(4)
    .reset_index(drop=True)
)


# ============================================================
# 7. RUN INTEGRATED GRADIENTS
# ============================================================

print("\n" + "=" * 80)
print("Generating Integrated Gradients explanations...")
print("=" * 80)

run_integrated_gradients()


# ============================================================
# 8. TRAINING CURVES
# ============================================================

# Re-create dataframe in case history was loaded
# from a checkpoint.

h = pd.DataFrame(history)

fig, ax = plt.subplots(
    1,
    3,
    figsize=(15, 3.5)
)


# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------

ax[0].plot(
    h["epoch"],
    h["train_loss"],
    label="Train"
)

ax[0].plot(
    h["epoch"],
    h["val_loss"],
    label="Validation"
)

ax[0].set_title(
    "Loss"
)

ax[0].set_xlabel(
    "Epoch"
)

ax[0].set_ylabel(
    "Loss"
)

ax[0].legend()


# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------

ax[1].plot(
    h["epoch"],
    h["train_acc"],
    label="Train"
)

ax[1].plot(
    h["epoch"],
    h["val_acc"],
    label="Validation"
)

ax[1].set_title(
    "Accuracy"
)

ax[1].set_xlabel(
    "Epoch"
)

ax[1].set_ylabel(
    "Accuracy"
)

ax[1].legend()


# ------------------------------------------------------------
# Macro F1
# ------------------------------------------------------------

ax[2].plot(
    h["epoch"],
    h["train_macro_f1"],
    label="Train"
)

ax[2].plot(
    h["epoch"],
    h["val_macro_f1"],
    label="Validation"
)

# Mark best epoch
ax[2].scatter(
    [best_epoch],
    [best_macro_f1],
    s=80,
    zorder=5,
    label=f"Best = {best_macro_f1:.4f}"
)

ax[2].set_title(
    "Macro F1"
)

ax[2].set_xlabel(
    "Epoch"
)

ax[2].set_ylabel(
    "Macro F1"
)

ax[2].legend()


plt.tight_layout()

plt.show()


# ============================================================
# 9. PRINT BEST-EPOCH INFORMATION
# ============================================================

print("\n" + "#" * 80)
print("TRAINING SUMMARY")
print("#" * 80)

print(
    f"Best Epoch           : {best_epoch}"
)

print(
    f"Best Validation F1   : {best_macro_f1:.4f}"
)

print(
    f"Test Accuracy        : {te['acc']:.4f}"
)

print(
    f"Test Balanced Acc    : {te['bal_acc']:.4f}"
)

print(
    f"Test Macro-F1        : {te['macro_f1']:.4f}"
)

print(
    f"Best model checkpoint: {BEST_MODEL_PATH}"
)

print("#" * 80)


# ============================================================
# 10. OPTIONAL: DOWNLOAD BEST MODEL
# ============================================================

# Uncomment if you want to download the trained model
#
# from google.colab import files
# files.download(BEST_MODEL_PATH)

Loading best model...

BEST VALIDATION RESULT
Best Epoch       : 3
Best Val Macro-F1: 0.5655


KeyboardInterrupt: 